In [1]:
#Run this cell for markdown formatting assistance. This is not a part of the assignment.
from IPython.core.display import HTML
table_css = 'table {align:left;display:block} '
HTML('<style>{}</style>'.format(table_css))

# DSE 230: Programming Assignment 2 – Word Count

## Submission on Gradescope

You must submit the following **FIVE files** under **“PA2”** on Gradescope:

1. **PA2_Starter.ipynb**  
   - The completed notebook with **all cells executed**  
   - All required outputs must be present in the designated cells  
   - Pass <code>truncate=False</code> as argument to <code>DataFrame.show</code> </li>

2. **100_words.csv**  
   - CSV file containing the **100 most frequently occurring words ending in `ing`**  
   - Must have exactly two columns named:
     - `word`
     - `count`

3. **exec_times.csv**  
   - CSV file containing execution times for **1, 2, and 4 cores**  
   - Must include:
     - Three trials per core setting  
     - The **average** execution time  
     - The **standard deviation**

4. **AI_chat_logs.pdf**  
   - PDF showing how you used AI tool:
     - Screenshots or saved chat logs  
     - Each interaction must include:
       - The **prompt you used**
       - The **AI’s response**
       - The **cell number** the interaction corresponds to  
   - **If you did NOT use an AI tool**, submit a PDF that clearly states:  
     > *“I did not use an AI tool for this assignment.”*

5. **AI_reflection.pdf**  
   - PDF containing a **1–2 paragraph reflection (150–300 words)** on how you used an AI tool and how useful was it
   - OR, if you did not use AI, a **brief statement** explaining your approach to completing the assignment independently  

---

#### IMPORTANT submission guidelines enforced by autograder. Please read carefully:
  * Make sure that all the cells in this notebook are executed and that the outputs are present in the expected cells before submission
  * Some cells are marked **DO NOT DELETE**. These cells cannot be deleted and the output of these cells will be used for autograding
  * You can add additional cells, but the **Expected Output** for each of the tasks MUST be the output of the cells marked as such
  * DO NOT print anything other than the *exact* expected output. **Do not include any sentences/words describing the output**. This is strictly enforced by the autograder which checks for an *exact* match of the expected output. For example, if you are expected to print the PySpark version:
      * '10.9.8' - <span style="color:#093">CORRECT</span>
      * 'The PySpark version is 10.9.8' - <span style="color:#FF0000">INCORRECT</span>
  * You can add cells for printing debugging information anywhere, but do not print anything else in **Expected Output** cells other than the expected output for the task
---

## Working on the Assignment

- Attempt each task **on your own first**, working **cell by cell**  
- If you get stuck, you may use an AI tool such as:
  - ChatGPT  
  - Claude  
  - VSCode GitHub Copilot  
- **AI usage must be documented carefully** 


---
Remember: when in doubt, read the documentation first. It's always helpful to search for the class that you're trying to work with, e.g. pyspark.sql.DataFrame. 

PySpark API Documentation: https://spark.apache.org/docs/4.0.1/api/python/index.html

Spark DataFrame Guide:  https://spark.apache.org/docs/4.0.1/sql-programming-guide.htmll


In [2]:
# Suppress native-hadoop warning
!sed -i '$a\# Add the line for suppressing the NativeCodeLoader warning \nlog4j.logger.org.apache.hadoop.util.NativeCodeLoader=ERROR,console' /$HADOOP_HOME/etc/hadoop/log4j.properties
# Suppress java warnings
import os; os.close(os.dup2(os.open(os.devnull, os.O_WRONLY), 2))

### 1. Copy data file `BookReviews_1M.txt` to the root of HDFS

This step is similar to Programming Assignment 1

#### **Expected output**: None

### 2. Start Spark Session

##### Change the number of cores in for your program by setting `spark.master` to `local[n]` in this code block where n take values 1,2 and 4.

#### **Expected output**: None

In [3]:
# Change the number of cores in this code block
# by setting `spark.master` to `local[n]` where
# n is the number of cores
import pyspark
from pyspark.sql import SparkSession

conf = pyspark.SparkConf().setAll([('spark.master', 'local[4]'),
                                   ('spark.app.name', 'Basic Setup')])
spark = SparkSession.builder.config(conf=conf).getOrCreate()

In [4]:
# Record the starting time of execution for timing this notebook
import time
start_time = time.time()

### 3. Load Data

Read data from the `BookReviews_1M.txt` file

#### **Expected output**: None

In [5]:
# DO NOT DELETE THIS CELL
df = spark.read.text("hdfs:///BookReviews_1M.txt")

### 4. Clean the data - 1 point

**Task:** Remove all punctuation and convert all characters to lower case.

Use the provided `removePunctuation` function (in the cell below) together with `DataFrame.select()` to apply the cleaning transformation to the `value` column of `textDF`.

**Save the result as a new dataframe called `cleanTextDF`.** This is the dataframe you will use in all following steps.

**Expected output:** The first 25 rows of `cleanTextDF`, showing the full cleaned sentences under a column named `sentence`. Pass `truncate=False` as an argument to `DataFrame.show` to display the entire sentence.

#### Your output would look like this, but the entire sentence:

|            sentence|
----------------------
|this was the firs...|
|also after going ...|
|as with all of ms...|
|ive not read any ...|
|this romance nove...|
|carolina garcia a...|
|not only can she ...|
|once again garcia...|
|the timing is jus...|
|engaging dark rea...|
|set amid the back...|
|this novel is a d...|
|if readers are ad...|
| reviewed by phyllis|
|      apooo bookclub|
|a guilty pleasure...|
|in the tradition ...|
|beryl unger top e...|
|what follows is a...|
|the book flap say...|
|id never before r...|
|the novels narrat...|
|it is centered on...|
|if you like moder...|
|beryl unger is a ...|

only showing top 25 rows

**NOTE** - The above table with cleaned sentences is for illustration only. Your output may differ slightly.

In [6]:
# We provide the following function for building a column expression for Task 1.
# Do not change this cell.

# NOTE: Counterintuitively, column objects do NOT store any data; instead they store column expressions (transformations).
#       The below function takes in a column object, and adds more expressions to it to make a more complex transformation.
#       Once we have a column object representing the expressions we want, use DataFrame.select(column) to apply the expressions

from pyspark.sql.functions import regexp_replace, trim, col, lower
def removePunctuation(column):
    """Removes punctuation, changes to lower case, and strips leading and trailing spaces."""
    return trim(lower(regexp_replace(column, "[^A-Za-z0-9 ]", ""))).alias("sentence")

In [7]:
# Recommended: take a look at the contents of a column object returned from removePunctuations. What's in there?
print(removePunctuation(df.value))

Column<'trim(lower(regexp_replace(value, '[^A-Za-z0-9 ]', ''))) AS sentence'>


#### **Expected output**: The first 25 rows of the cleaned dataframe, with a column containing the **entire** cleaned sentences, under a column named `sentence`

#### Pass `truncate=False` as argument to `DataFrame.show`.

In [8]:
# DO NOT DELETE THIS CELL
cleanTextDF = df.select(removePunctuation(col("value")))
cleanTextDF.show(25, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|sentence                                                                                                         

### 5. Get a dataframe of unique words ending in 'ing' and their counts - 3 points

**You will work with `cleanTextDF` (the cleaned dataframe from Step 4) throughout this entire step.**

#### 5.1 Create a dataframe of words - 1 point

Starting from `cleanTextDF`, do the following **in order**:

a. **Split** each sentence into individual words using a single space `' '` as the delimiter.  
b. **Explode** the resulting list so that each word gets its own row.  

Save the result as a new dataframe with a **single column named `word`**.

> Useful functions: [`pyspark.sql.functions.split`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html), [`pyspark.sql.functions.explode`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html)

**Expected output:** The first 5 rows of the resulting dataframe, with a single column named `word`.

#### **Expected output**: The first 5 rows of the resulting dataframe, with a single column named `word`

#### Pass `truncate=False` as argument to `DataFrame.show`.

In [9]:
# DO NOT DELETE THIS CELL
from pyspark.sql.functions import split, explode

wordsDF = cleanTextDF.select(split(col("sentence"), " ").alias("words"))
wordsDF = wordsDF.select(explode(col("words")).alias("word"))
wordsDF.show(5, truncate=False)

+-----+
|word |
+-----+
|this |
|was  |
|the  |
|first|
|time |
+-----+
only showing top 5 rows


#### 5.2 Filter words that end in 'ing' and count them - 2 points

#### Tasks:

<ol>
    <li type = "a"> Filter the dataframe to contain only words that end in 'ing'. </li>
    <li type = "a"> Group rows in the previous dataframe by unique words, then count the rows in each group.
</ol>

#### Expected output:

<ol>
    <li type = "a"> First 20 rows of the dataframe, where each row contains only one word that ends in 'ing'. The dataframe must not contain empty rows. </li>
    <li type = "a"> First 20 rows of the dataframe containing unique words and their counts. </li>
</ol>

##### The output after filtering words ending in 'ing' would look like this:

|          word|
----------------
|reading          |
|looking         |
|feeling           |

... 17 more

##### The output after grouping unique words and their counts would look like this:

|       word|count|
------------|------
|reading    |66968  |
|looking |50000    |
|something  |20000 |

... 17 more

**NOTE** - The above table with words and counts is for illustration only. Your output should contain all 20 rows for each of the tasks, and your counts may differ.

#### **Expected output**: The first 20 rows of a dataframe, where each row contains only one word ending in 'ing', under a column named `word`.
#### Pass `truncate=False` as argument to `DataFrame.show`.

In [10]:
# DO NOT DELETE THIS CELL
ingDF = wordsDF.filter(col("word").endswith("ing") & (col("word") != ""))
ingDF.show(20, truncate=False)

+-----------+
|word       |
+-----------+
|looking    |
|coming     |
|looking    |
|going      |
|feeling    |
|having     |
|going      |
|amazing    |
|being      |
|reading    |
|intriguing |
|turning    |
|interesting|
|timing     |
|something  |
|engaging   |
|engaging   |
|reading    |
|stimulating|
|writing    |
+-----------+
only showing top 20 rows


#### **Expected output**: First 20 rows of the dataframe containing unique words that end in 'ing' and their counts, under columns named `word` and `count` respectively
#### Pass `truncate=False` as argument to `DataFrame.show`.

In [11]:
# DO NOT DELETE THIS CELL
ingCountDF = ingDF.groupBy("word").count()
ingCountDF.show(20, truncate=False)

+---------------+-----+
|word           |count|
+---------------+-----+
|involving      |142  |
|incoming       |341  |
|traveling      |2210 |
|distancing     |5    |
|spoiling       |10   |
|blanking       |16   |
|safeguarding   |4    |
|filing         |110  |
|biting         |34   |
|concluding     |20   |
|resolvednothing|1    |
|percieving     |1    |
|inverting      |6    |
|scrapping      |14   |
|affixing       |25   |
|squealing      |50   |
|tripping       |219  |
|brandssaving   |1    |
|balding        |3    |
|varrying       |1    |
+---------------+-----+
only showing top 20 rows


### 6. Sort the word count dataframe in a **descending** order by count - 1 point

**CHECK** - The first row would have the maximum count.

#### **Expected output**: First 20 rows of the word count dataframe sorted in **descending** order by counts, with columns named `word` and `count`

#### Pass `truncate=False` as argument to `DataFrame.show`.

In [12]:
# DO NOT DELETE THIS CELL
sortedDF = ingCountDF.orderBy("count", ascending=False)
sortedDF.show(20, truncate=False)

+----------+-----+
|word      |count|
+----------+-----+
|using     |58806|
|thing     |35578|
|looking   |31521|
|something |26517|
|going     |25458|
|working   |24984|
|being     |22124|
|getting   |21941|
|having    |21796|
|everything|21300|
|anything  |17828|
|buying    |16898|
|nothing   |15197|
|running   |13673|
|amazing   |13278|
|shipping  |11711|
|trying    |11682|
|listening |11629|
|reading   |9871 |
|taking    |9620 |
+----------+-----+
only showing top 20 rows


### 7. Record the execution time

#### **Expected output**: The execution time. No particular value is expected. This will be needed in section 10.

In [13]:
print(time.time() - start_time)

60.68785095214844



### 8. Save the sorted word counts to HDFS as a CSV file - 1 point

**NOTE**: Spark uses a distributed memory system, and stores working data in fragments known as "partitions". This is advantageous when a Spark cluster spans multiple machines, as each machine will only require part of the working data to do its own job. By default, Spark will save each of these data partitions into a individual file to avoid I/O collisions. We want only one output file, so we'll need to fuse all the data into a single partition first. 

Your task: 
1. Coalesce the previous dataframe to one partition using `DataFrame.coalesce(1)`. This returns a 1-partition dataframe. This makes sure that all our results will end up in the same csv file. 
2. Save the 1-partition dataframe to HDFS using the `DataFrame.write.csv(<path>)` method to the root directory of the HDFS, i.e. `hdfs:///<<your-result-file>>.csv`.

#### **Expected output**: None

In [14]:
# YOUR CODE HERE TO SAVE THE DATAFRAME AS A CSV FILE
sortedDF.coalesce(1).write.csv("hdfs:///wordcount_results.csv", header=True, mode="overwrite")

The resultant file saved in the step above is actually a folder, which contains individually saved files from each partition of the saved dataframe. <br> <br>
Now, use an HDFS command to show the contents of the resulting folder on HDFS from the last step in the cell below. <br>
You will need to include ‘!’ before the HDFS command for Jupyter Notebook to recognize it as an operating system command. 

#### **Expected output**: List of files in the result directory

In [15]:
# DO NOT DELETE THIS CELL
# OS command to show the generated file(s)
!hdfs dfs -ls /

Found 2 items
-rw-r--r--   1 root supergroup  219041604 2026-04-29 03:45 /BookReviews_1M.txt
drwxr-xr-x   - root supergroup          0 2026-04-30 04:21 /wordcount_results.csv


Run the code below to see the working directory on your local system. The exclamation point '!' is to specify that it is an Operating System command, and 'pwd' is a LINUX command for 'print working directory'.

#### **Expected output**: Working directory on your local system

In [16]:
! pwd

/home


Now, stop the spark session in the cell below
#### **Expected output**: None

In [17]:
# DO NOT DELETE THIS CELL
spark.stop()


### 9. Copy the results from HDFS to the local file system - 1 point

Now that we have our results stored in HDFS, we need to copy it back to the local file system to access it. This process may sound cumbersome, but it is a necessary result of Spark and Hadoop's distributed architecture, and their ability to scale up to arbitrarily large datasets and computing operations. 

Copying the results from HDFS to the local file system:
1. Run an hdfs command in the terminal to list the root directory of the HDFS. You should see the CSV file that you have saved. Counterintuitively, this CSV file is actually a folder, which contains individually saved files from each partition of the saved dataframe (see above for data partitioning).  
2. Run another hdfs command to see what's inside the saved folder. Since we made sure to coalesce our dataframe to just one partition, we should expect to find only one saved partition in this folder, saved also as a CSV. Note the name of this file, it should look something like `part-00000-xx.....xx.csv`. 
3. Now copy the resultant CSV file from HDFS to the current folder on your local file system using an hdfs command in the terminal. You may rename this file to something more interpretable - let's say `results.csv`. 
4. We want you to submit a CSV containing the first 101 rows of the results file. To do this, use the command `head -n 101 results.csv > 100_words.csv`. You can also do so manually, since CSV files are in plain text. Remember that we want the first 101 lines which would include the header as well - so basically it is header + 100 rows.

#### **Expected output**: None

### 10. Submission of `exec_times.csv` containing execution times on different number of cores - 1 point

#### **Expected output**(in exec_times.csv file) - Execution times on 1,2 and 4 cores, 3 trials for each core count, the mean and standard deviation of execution times for each core count. The submission should follow the exact template shown below

**NOTE** - No output is expected in the notebook

After writing all of the expected code before this cell, you should set the configuration at the beginning of this Notebook in the cell where this code is present:

```conf = pyspark.SparkConf().setAll([('spark.master', 'local[1]'), ('spark.app.name', 'Word Count')])```

Create a csv file `exec_times.csv` and fill it with the following template:
```


| Cores | Runtime_1 | Runtime_2 | Runtime_3 | Mean | Std |
--------|-----------|-----------|-----------|------|------
|1| 95.63300132751465| 94.7302520275116| 94.6054916381836| 94.9895816644033| 0.560698614092323|... |
|2| 66.85855507850647| 66.65802145004272| 66.07460832595825| 66.5303949515024| 0.407258542028224|... |
|4| 58.612706899642944| 58.26924657821655| 58.503124475479126| 58.4616926511128| 0.175438579426872|... |




### 11. Submission of `100_words.csv` - 1 point

#### **Expected output**(in the 100_words.csv file) - Top 100 unique words ending in 'ing' and their counts, sorted in descending order.

**NOTE** - No output is expected in the notebook

The csv file should have two columns, `word` and `count`, in the first line and 100 more lines with the top 100 unique words ending in 'ing' and their counts, sorted in descending order of the counts.

### Note on Autograder

The autograder will check whether the results that you submit in the `100_words.csv` file matches **exactly** with the expected results or not.

The csv file would look something like this:

|       word|count|
------------|------
|       really| 66968|
|       very| 50000|
|      already|45000|

... 97 more

The counts are shown for illustration -- Your counts may differ

### 12. Submission of `AI_reflection.pdf` - 1 point

#### **Expected output** - A PDF document reflecting on your use of AI tools during this assignment.

**NOTE** - No output is expected in the notebook

The PDF file should be named `AI_reflection.pdf` and should include the following sections:

### Reflect on Your Experience

Write **1–2 paragraphs (150–300 words)** addressing:

- Specific errors or challenges the AI helped you debug  
- How the interaction improved your understanding of **PySpark** or coding  
- What you learned from working with the AI tool  
